# 🧪 StockGen AI - PyTorch Baseline Performance Test
สมุดโน้ตเล่มนี้ใช้สำหรับทดสอบประสิทธิภาพ **Baseline** ของโมเดล Real-ESRGAN บน Google Colab (GPU Tesla T4) เพื่อบันทึกสถิติความเร็วในการประมวลผลก่อนนำไปเปรียบเทียบกับระบบจริงบน RunPod และการทำ TensorRT Optimization ในอนาคตครับ

### 🚀 วิธีใช้งาน:
1. ไปที่เมนู **Runtime** -> **Change runtime type** ตรวจสอบว่าเป็น **GPU** (Tesla T4)
2. กดปุ่ม **Runtime** -> **Run all** เพื่อเริ่มการทดสอบทั้งหมด

### 📦 สเต็ปที่ 1: ติดตั้งไลบรารีและ Patch ระบบ
ติดตั้งโปรแกรมที่จำเป็นและแก้ไขปัญหาความเข้ากันได้ของไลบรารีพื้นฐาน

In [ ]:
!pip install realesrgan gfpgan basicsr opencv-python tqdm
import sys, os, types, torch, time, cv2, numpy as np
from tqdm import tqdm
import torchvision.transforms.functional as F_tv

print("⏳ กำลังตั้งค่า Patch เพื่อความเข้ากันได้...")
# Patch basicsr สำหรับ PyTorch รุ่นใหม่
try:
    shim = types.ModuleType("torchvision.transforms.functional_tensor")
    shim.rgb_to_grayscale = F_tv.rgb_to_grayscale
    sys.modules["torchvision.transforms.functional_tensor"] = shim
    print("✅ Environment Ready!")
except Exception as e:
    print(f"⚠️ Patch Warning: {e}")

### 🧠 สเต็ปที่ 2: เตรียมโมเดลและฟังก์ชัน Benchmark
กำหนดค่าโมเดลหลัก 3 ตัว (x2, x4, Anime) และฟังก์ชันสำหรับวัดความเร็วเฉลี่ย

In [ ]:
from realesrgan import RealESRGANer
from basicsr.archs.rrdbnet_arch import RRDBNet

def get_upsampler(model_type='x2plus'):
    print(f"📂 กำลังโหลดโมเดล: {model_type}...")
    if model_type == 'x2plus':
        model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=2)
        url = 'https://huggingface.co/nateraw/real-esrgan/resolve/main/RealESRGAN_x2plus.pth'
        scale = 2
    elif model_type == 'x4plus':
        model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=4)
        url = 'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth'
        scale = 4
    elif model_type == 'anime':
        model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=6, num_grow_ch=32, scale=4)
        url = 'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.2.4/RealESRGAN_x4plus_anime_6B.pth'
        scale = 4

    return RealESRGANer(scale=scale, model_path=url, model=model, tile=0, half=True, device='cuda')

def benchmark(model_type, width, height, iterations=3):
    upsampler = get_upsampler(model_type)
    # สร้างภาพจำลอง (Random Noise)
    dummy_img = np.random.randint(0, 255, (height, width, 3), dtype=np.uint8)
    
    # Warm-up ครั้งแรก (เพื่อให้ GPU พร้อมทำงานเต็มที่)
    print(f"🔥 Warming up {model_type}...")
    upsampler.enhance(dummy_img, outscale=upsampler.scale)
    
    print(f"🚀 เริ่มทดสอบ {iterations} รอบ...")
    start_time = time.time()
    for i in range(iterations):
        t_start = time.time()
        upsampler.enhance(dummy_img, outscale=upsampler.scale)
        t_end = time.time()
        print(f"   - รอบที่ {i+1}: {t_end - t_start:.2f}s")
    
    avg_time = (time.time() - start_time) / iterations
    print(f"✅ เฉลี่ย: {avg_time:.2f}s\n")
    return avg_time

### 📊 สเต็ปที่ 3: รันการทดสอบและสรุปผล
ทดสอบขนาดภาพยอดนิยม (512px และ 1024px) บนโมเดลแต่ละประเภท

In [ ]:
results = {}
test_sizes = [(512, 512), (1024, 1024)]
models_to_test = ['x2plus', 'x4plus', 'anime']

print("🏁 เริ่มกระบวนการ Benchmark บน Tesla T4 GPU...\n")

for model in models_to_test:
    results[model] = {}
    for size in test_sizes:
        avg = benchmark(model, size[0], size[1])
        results[model][f"{size[0]}x{size[1]}"] = f"{avg:.2f}s"

print("--- 🏁 สรุปผลการทดสอบ (Benchmark Summary) ---")
import json
print(json.dumps(results, indent=4))

print("\n💡 รบกวนคัดลอกผลลัพธ์ JSON ด้านบนกลับไปบอก Gemini CLI เพื่อบันทึกสถิติครับ!")